# Skeleton Lab: Bayes' Theorem, LDA & QDA — Practice From Scratch

Fill in every `# TODO` below. Work top to bottom — later parts assume earlier variables exist.
Check your work against `Solutions_LDA_QDA_Lab.ipynb` only after you've made a genuine attempt.

**Datasets:** `affairs` (via `statsmodels`) for LDA, `iris` (via `sklearn`) for QDA, plus one
synthetic dataset you will generate yourself in Part F.

## Checklist of what you'll build
- [ ] Part A: Bayes' Theorem update function
- [ ] Part B: LDA pipeline on `affairs` (EDA → assumption checks → preprocessing → fit → evaluate)
- [ ] Part C: LDA for supervised dimensionality reduction
- [ ] Part D: QDA pipeline on `iris` (fit → evaluate → visualize decision boundary)
- [ ] Part E: Compare Logistic Regression vs LDA vs QDA
- [ ] Part F: Build a synthetic dataset where QDA should beat LDA, and prove it
- [ ] Part G: Cross-validate your models


## Setup
Run this cell as-is.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                              precision_score, recall_score, roc_auc_score, roc_curve,
                              classification_report)
from sklearn.datasets import load_iris

np.random.seed(42)
sns.set_style("whitegrid")


---
## Part A — Bayes' Theorem from scratch

Write a function that computes $P(\text{sick}\mid\text{positive test})$ given:
- `prior_sick`: prior probability someone is sick
- `true_positive_rate`: P(positive test | sick)
- `false_positive_rate`: P(positive test | not sick)

Then use it to show how the posterior changes after 1, 2, and 3 consecutive positive tests,
starting from a prior of 0.2, a true-positive rate of 0.95, and a false-positive rate of 0.3.

In [ ]:
def posterior_sick_given_positive(prior_sick, true_positive_rate, false_positive_rate):
    # TODO: implement Bayes' Theorem here.
    # Hint: P(positive) = P(pos|sick)*P(sick) + P(pos|not sick)*P(not sick)
    pass

# TODO: compute and print the posterior after 1, 2, and 3 consecutive positive tests


In [ ]:
# TODO: plot how the belief (posterior) evolves over ~6 consecutive positive tests
# Use a line plot: x = number of tests, y = P(sick | evidence so far)


---
## Part B — LDA on the `affairs` dataset

### B1. Load and label the data
Load the `affairs` dataset from `statsmodels` (`sm.datasets.fair.load().data`). Recode the
`affairs` column so any value greater than 0 becomes `1`, and 0 stays `0`.

In [ ]:
import statsmodels.api as sm

# TODO: load df_affairs
# TODO: recode df_affairs['affairs'] to binary (0/1)


In [ ]:
# TODO: report the class balance (% class 0 vs % class 1)


In [ ]:
# TODO: build X (features) and y (target).
# Features to use: rate_marriage, age, yrs_married, children, religious, educ, occupation, occupation_husb
numeric_cols = ['rate_marriage', 'age', 'yrs_married', 'children', 'religious', 'educ']


### B2. Check the LDA assumptions
Check (a) approximate normality of each numeric feature per class, and (b) approximate equal covariance across classes. You can reuse Shapiro-Wilk / KDE plots for (a) and a Frobenius-norm covariance comparison for (b) — or design your own approach.

In [ ]:
# TODO: for each numeric column, plot the class-0 vs class-1 distribution (KDE or histogram)
# and optionally run a normality test (e.g. scipy.stats.shapiro)


In [ ]:
# TODO: compute the per-class covariance matrices for the numeric columns and compare them
# (e.g. Frobenius norm of the difference, or just eyeball two heatmaps side by side)


### B3. Preprocess
One-hot encode `occupation`, drop `occupation_husb`, split into train/test (33% test, stratified, `random_state=42`), then scale the numeric columns.

**Careful:** fit the scaler only on the training data, then just `.transform()` the test data — don't fit a second scaler on the test set (that's a data-leakage bug).

In [ ]:
# TODO: map occupation codes to readable strings and one-hot encode
# TODO: drop occupation_husb
# TODO: train_test_split (test_size=0.33, random_state=42, stratify=y)
# TODO: fit a StandardScaler on numeric_cols using ONLY the training data;
#       transform both train and test with that same fitted scaler


### B4. Fit and evaluate LDA
Fit `LinearDiscriminantAnalysis`. Report precision, recall, and accuracy on both train and test sets, plus confusion matrices for both.

In [ ]:
# TODO: fit LDA on the scaled training data
# TODO: predict on train and test
# TODO: print precision / recall / accuracy for both


In [ ]:
# TODO: plot confusion matrices for train and test side by side


### B5. ROC curve and AUC
Compute predicted probabilities for the test set and plot an ROC curve with the AUC in the legend.

In [ ]:
# TODO: get predict_proba for the positive class on X_test_sc
# TODO: compute fpr, tpr, and AUC; plot the ROC curve


**Your interpretation:** In a markdown cell, write 1-2 sentences on what the AUC tells you about how strong this model actually is.

---
## Part C — LDA for supervised dimensionality reduction

Use `LinearDiscriminantAnalysis.fit_transform()` on the training data to reduce it to its
LDA-discriminant axis (or axes). Confirm the new shape, and plot the class-conditional density
along the first (and only, since there are 2 classes) discriminant axis.

In [ ]:
# TODO: fit_transform LDA on X_train_sc, y_train to get X_reduced
# TODO: print the shape before and after reduction


In [ ]:
# TODO: plot KDE of X_reduced separately for class 0 and class 1


---
## Part D — QDA on the `iris` dataset

### D1. Load, split, fit
Load `iris` from `sklearn.datasets`, build a DataFrame with readable column names, split
80/20 (stratified, `random_state=1`), fit `QuadraticDiscriminantAnalysis`, and report test
accuracy plus a full classification report.

In [ ]:
# TODO: load iris, build df_iris with columns sepal_length, sepal_width, petal_length,
#       petal_width, target, species


In [ ]:
# TODO: train/test split (80/20, stratified, random_state=1)
# TODO: fit QuadraticDiscriminantAnalysis(store_covariance=True)
# TODO: predict on test set, print accuracy and classification_report


In [ ]:
# TODO: plot the confusion matrix (use ConfusionMatrixDisplay with iris.target_names as labels)


### D2. 2D visualization
Using only `sepal_length` and `sepal_width`, fit a fresh QDA model, build a mesh grid over
the feature ranges, predict on the grid, and plot filled contour regions with the true points overlaid.

In [ ]:
# TODO: build df1 with just sepal_length, sepal_width, target, species
# TODO: fit a 2-feature QDA model
# TODO: build a meshgrid over sepal_length in [4, 8] and sepal_width in [1.5, 4.5], 400+ points each
# TODO: predict class over the grid, reshape, and plot contourf + contour + scatter of true points


---
## Part E — Model shootout: Logistic Regression vs LDA vs QDA

Fit all three models on the same 2D iris slice (`sepal_length`, `sepal_width`) and plot their
decision boundaries side by side in a single figure (1 row, 3 columns). Title each subplot with
the model name and its training accuracy.

In [ ]:
# TODO: fit LogisticRegression, LinearDiscriminantAnalysis, and QuadraticDiscriminantAnalysis
#       on the same 2D data
# TODO: plot decision boundaries for all three, side by side


**Your interpretation:** Which boundaries are straight lines and which curve? Why?

---
## Part F — Build your own unequal-covariance dataset

Generate two synthetic 2D Gaussian classes using `numpy.random.multivariate_normal`:
- Class A: mean `[0, 0]`, a small, round (uncorrelated, low-variance) covariance matrix.
- Class B: mean `[3, 3]`, a larger covariance matrix with meaningful positive correlation
  between the two dimensions.

Use ~300 points per class. Split into train/test, fit both LDA and QDA, and compare test
accuracy. Then plot both models' decision boundaries side by side over the scattered points.

In [ ]:
# TODO: generate class A and class B with numpy.random.multivariate_normal
# TODO: stack into X_syn, y_syn; train_test_split


In [ ]:
# TODO: fit LDA and QDA on the synthetic training data; print test accuracy for both


In [ ]:
# TODO: plot decision boundaries for both models side by side, with the two classes scattered on top


**Your interpretation:** Which model wins, and does that match your prediction from Part D/E about when QDA should outperform LDA?

---
## Part G — Cross-validate your models

Run 5-fold stratified cross-validation for:
1. LDA on the full (scaled) `affairs` dataset, scoring on `roc_auc`.
2. QDA on the full `iris` dataset, scoring on `accuracy`.

Report the mean and standard deviation of each.

In [ ]:
# TODO: build a StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# TODO: cross_val_score for LDA on affairs (roc_auc)
# TODO: cross_val_score for QDA on iris (accuracy)
# TODO: print mean +/- std for both


---
## Wrap-up
In a markdown cell, write 3-5 bullet points summarizing:
- Which assumption checks passed or failed, and what you'd do differently.
- Whether LDA or QDA is the better fit for the `affairs` data, and why.
- One real-world dataset (not in this lab) where you'd expect LDA to work well, and one where
  you'd expect it to fail.
